# Writing Reliable Code

Chapters 17 and 18 wrote code that depends on somebody else's server being up, returning what it returned last week, and keeping its HTML where it was. That is exactly the code most likely to break while you are asleep, and the code you will least enjoy debugging six months from now.

This chapter is about the four things that make the difference, and none of them are about making the program work. It already works. They are about what happens next:

- **Saying what you mean**, with type hints and docstrings, so the next reader does not have to guess. The next reader is usually you.
- **Saying what happened**, with `logging` rather than `print`, so a failure at 3am leaves evidence.
- **Proving it works**, with tests, so a change you make in March does not quietly break something you wrote in January.
- **Finding out why**, by reading tracebacks properly and using a debugger.

Two of these were promised earlier and never delivered: chapter 9 listed `assert` and logging among the things it would cover and ran out of room. They are here.

**What we will learn:**

1. Type hints, and the fact that Python ignores them
2. Running a type checker, which does not
3. Docstrings, and what belongs in one
4. `logging`: levels, loggers, and why `print` is not enough
5. `assert` against real validation, and the flag that deletes your asserts
6. `pytest`: writing tests, running them, reading a failure
7. Reading a traceback, and `breakpoint()`
8. PEP 8, briefly

### Setting Up

This chapter writes a small package and tests it, so it needs a folder to work in. `shutil.rmtree` clears out anything left from a previous run, which keeps the output the same every time.

In [1]:
import os
import shutil
from pathlib import Path

shutil.rmtree("reliable_demo", ignore_errors=True)
os.makedirs("reliable_demo", exist_ok=True)

print("working in", Path("reliable_demo").resolve().name)

working in reliable_demo


---
# 1. Type Hints

A type hint says what a function expects and what it gives back. It goes after a colon for arguments, and after an arrow for the return value.

```python
def average(numbers: list[float]) -> float:
    return sum(numbers) / len(numbers)
```

Everything in this course so far has been written without them, and it all ran. So what are they for?

### Python does not check them

This is the first thing to understand, and it surprises nearly everybody:

In [2]:
def average(numbers: list[float]) -> float:
    return sum(numbers) / len(numbers)


print(average([10.0, 20.0, 30.0]))
print(average([1, 2, 3]))                   # ints, not floats. Fine.
print(average((4, 5, 6)))                   # a tuple, not a list. Also fine.

20.0
2.0
5.0


None of those match the hint, and Python did not care. Annotations are **not enforced at runtime**. They are recorded on the function and otherwise ignored:

In [3]:
print(average.__annotations__)

{'numbers': list[float], 'return': <class 'float'>}


So a type hint is not a safety net. It is two other things.

**It is documentation that cannot drift.** A comment saying "pass a list of floats" goes stale the moment someone changes the function. A hint sits in the signature where the change happens.

**It is checkable by a tool.** Which is the next section, and the reason hints are worth writing at all.

### The syntax you will actually meet

| Hint | Means |
|---|---|
| `x: int` | an integer |
| `x: str` | a string |
| `x: list[int]` | a list of integers |
| `x: dict[str, float]` | a dict with string keys and float values |
| `x: tuple[str, int]` | a tuple of exactly those two |
| `x: str \| None` | a string, or `None` |
| `x: list[dict[str, str]]` | nesting works |
| `-> None` | returns nothing useful |

`str | None` is the one you will see most, because it describes every function that might not find what it was looking for. Chapter 18's `text_of` is exactly that shape, so here it is with hints:

In [4]:
from bs4 import BeautifulSoup


def text_of(parent, selector: str, default: str | None = None) -> str | None:
    found = parent.select_one(selector)
    return found.get_text(strip=True) if found else default


soup = BeautifulSoup(Path("sample_data/shop_page_2.html").read_text(encoding="utf-8"),
                     "html.parser")
article = soup.select("article.product")[1]

print(repr(text_of(article, "h3 a")))
print(repr(text_of(article, "p.price")))
print(repr(text_of(article, "p.price", default="")))

'Effective Pandas'
None
''


The signature now tells you what the last line already showed: this function can hand you `None`, so check for it. That is the whole point. You will meet the same shape constantly the moment you read library code, where `Optional[float]` is the older spelling of `float | None`.

---
# 2. A Tool That Does Check

`mypy` reads the hints and reports the places where they do not add up, without running anything. Here is a file with two genuine mistakes in it.

In [5]:
%%writefile reliable_demo/prices.py
"""Turning scraped price text into numbers."""


def to_price(text: str) -> float | None:
    """Return the number in a price string, or None if there is not one."""
    import re
    digits = re.sub(r"[^\d.]", "", text or "")
    return float(digits) if digits else None


def total(prices: list[float]) -> float:
    """Add up a list of prices."""
    return sum(prices)

Writing reliable_demo/prices.py


In [6]:
%%writefile reliable_demo/uses_prices.py
from prices import to_price, total

total(["12.50", "8.00"])                # strings, not floats
count: int = to_price("£12.50")         # to_price can return None, and a float

Writing reliable_demo/uses_prices.py


In [7]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "mypy", "uses_prices.py",
     "--no-color-output", "--no-error-summary"],
    cwd="reliable_demo", capture_output=True, text=True)

print(result.stdout or "(mypy found nothing)")

uses_prices.py:3: error: List item 0 has incompatible type "str"; expected "float"  [list-item]
uses_prices.py:3: error: List item 1 has incompatible type "str"; expected "float"  [list-item]
uses_prices.py:4: error: Incompatible types in assignment (expression has type "float | None", variable has type "int")  [assignment]



Two mistakes, neither of which would raise anything on the line where it was written. The first would fail later inside `sum`. The second would sail on until something tried arithmetic on a `None`, which is the exact bug chapter 18's missing price would cause.

You run `mypy` the same way you run tests: on the command line, or automatically before a commit.

```bash
pip install mypy
mypy your_package/
```

Hints are optional and partial. Add them to the functions that other code calls, leave them off a three-line script, and do not let anyone tell you it is all or nothing.

---
# 3. Docstrings

A string as the first statement in a function, class or module is a **docstring**. Python stores it, `help()` prints it, and every editor shows it on hover.

In [8]:
def to_price(text: str) -> float | None:
    """Return the number in a price string, or None if there is not one.

    Strips currency symbols, spaces and thousands separators, so
    "£1,299.00" becomes 1299.0 and "free" becomes None.
    """
    import re
    digits = re.sub(r"[^\d.]", "", text or "")
    return float(digits) if digits else None


print(to_price.__doc__.splitlines()[0])
print()
help(to_price)

Return the number in a price string, or None if there is not one.

Help on function to_price in module __main__:

to_price(text: str) -> float | None
    Return the number in a price string, or None if there is not one.

    Strips currency symbols, spaces and thousands separators, so
    "£1,299.00" becomes 1299.0 and "free" becomes None.



The first line is a one-sentence summary, in the imperative: "Return the number", not "This function returns the number". Then a blank line, then anything a caller needs that the signature does not already say.

What belongs in one, and what does not:

| Put in | Leave out |
|---|---|
| What it returns, including the surprising cases | The types, if you have written hints |
| What it raises, and when | A restatement of the function name |
| Units, formats, assumptions | How it works internally |
| An example, if the call is not obvious | Change history, which git already has |

A docstring that says `"""Converts text to price."""` above `def to_price(text)` has added nothing. `"""...or None if there is not one."""` has told the caller something they had to know.

---
# 4. Logging

Every chapter so far has used `print` to show what code is doing. `print` is fine for a notebook and wrong for anything that runs unattended.

| | `print` | `logging` |
|---|---|---|
| Turn it off without editing code | No | Yes, by level |
| Say how important a message is | No | Five levels |
| Say where it came from | No | Logger name, usually the module |
| Send it to a file as well as the screen | No | Yes, several places at once |
| Timestamps | You write them yourself | Built in |

### The five levels

They exist so you can turn the volume down without deleting anything.

| Level | For |
|---|---|
| `DEBUG` | detail you want while developing |
| `INFO` | normal progress worth recording |
| `WARNING` | something unexpected, but it carried on |
| `ERROR` | it failed and gave up on this piece of work |
| `CRITICAL` | the program cannot continue |

In [9]:
import logging
import sys

logger = logging.getLogger("shop.scraper")          # a name, usually __name__
logger.setLevel(logging.DEBUG)
logger.handlers.clear()                             # so re-running this cell is clean

handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)-8s %(name)s | %(message)s"))
logger.addHandler(handler)
logger.propagate = False

logger.debug("parsed 3 products from page 1")
logger.info("page 1 of 3 done")
logger.warning("no price for sku 9781098139482")
logger.error("page 4 returned 500, giving up on it")
logger.critical("no pages could be fetched at all")

DEBUG    shop.scraper | parsed 3 products from page 1


INFO     shop.scraper | page 1 of 3 done


WARNING  shop.scraper | no price for sku 9781098139482


ERROR    shop.scraper | page 4 returned 500, giving up on it


CRITICAL shop.scraper | no pages could be fetched at all


In a real script the format string would start with `%(asctime)s` for a timestamp. This notebook leaves it out so the output is the same every time it runs.

### Turning the volume down

The level is a threshold. Raise it and the quieter messages disappear, with no code changed:

In [10]:
logger.setLevel(logging.WARNING)

logger.debug("parsed 3 products from page 1")       # gone
logger.info("page 1 of 3 done")                     # gone
logger.warning("no price for sku 9781098139482")
logger.error("page 4 returned 500, giving up on it")

print("\nonly two of the four messages got through")

WARNING  shop.scraper | no price for sku 9781098139482


ERROR    shop.scraper | page 4 returned 500, giving up on it



only two of the four messages got through


This is the argument in one cell. Every `print` you have ever deleted before shipping could have been a `logger.debug` you left in.

### One logger per module

The usual line at the top of a module is:

```python
logger = logging.getLogger(__name__)
```

`__name__` is the module's name, from chapter 11. In a package called `shop`, `shop/scraper.py` gets a logger called `shop.scraper`, and because loggers are hierarchical you can silence the whole package with one call on `shop`, or turn up just the scraper.

### Logging an exception

Inside an `except` block, `logger.exception()` records the message **and** the traceback. It is the single most useful call in the module.

In [11]:
logger.setLevel(logging.DEBUG)


def parse_row(row):
    return float(row["price"])


try:
    parse_row({"price": "not a number"})
except ValueError:
    logger.exception("could not parse row 12")

ERROR    shop.scraper | could not parse row 12
Traceback (most recent call last):
  File "/var/folders/6t/w0q7sf4n28n8p23_b3rqs2x80000gn/T/ipykernel_56049/1405366784.py", line 9, in <module>
    parse_row({"price": "not a number"})
  File "/var/folders/6t/w0q7sf4n28n8p23_b3rqs2x80000gn/T/ipykernel_56049/1405366784.py", line 5, in parse_row
    return float(row["price"])
           ^^^^^^^^^^^^^^^^^^^
ValueError: could not convert string to float: 'not a number'


The traceback is in the log, and the program carried on. (The file name in it is the temporary one Jupyter gives a notebook cell, which is why that cell is one of the few here whose output is not checked for reproducibility.) Compare that with `except ValueError: pass`, which is chapter 9's warning, or with `print("error!")`, which tells you nothing you can act on.

### Setting it up in a script

The three lines above are for demonstrating the pieces. A script uses `basicConfig` once, at the top of `main`, and every module's logger then feeds into it:

```python
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s",
    handlers=[logging.FileHandler("scraper.log"),
              logging.StreamHandler()],          # file and screen
)
```

Two rules worth having from the start. **Never log a secret**, because logs get copied, emailed and shipped to other services, which chapter 17 said about API keys. And **log the values you will want**, not just the fact of the failure: `"no price for sku %s"` with the sku is useful; `"parse failed"` is not.

---
# 5. `assert`, and the Flag That Deletes It

Chapter 9 mentioned `assert` and never came back to it. `assert condition, message` raises `AssertionError` when the condition is false.

In [12]:
def average_price(prices):
    assert prices, "average_price got an empty list"
    return sum(prices) / len(prices)


print(average_price([10.0, 20.0]))

try:
    average_price([])
except AssertionError as e:
    print("AssertionError:", e)

15.0
AssertionError: average_price got an empty list


That looks like a perfectly good way to validate input, and it is not, because of one flag.

Running Python with `-O` **removes every assert statement** from the compiled code. Not "skips": removes. Here is a function whose only guard is an assert:

In [13]:
%%writefile reliable_demo/guard.py
def withdraw(balance, amount):
    assert amount > 0, "amount must be positive"
    return balance - amount


print("balance after withdrawing -50:", withdraw(100, -50))

Writing reliable_demo/guard.py


In [14]:
for flags in ([], ["-O"]):
    result = subprocess.run([sys.executable, *flags, "guard.py"],
                            cwd="reliable_demo", capture_output=True, text=True)
    label = "python3 " + " ".join(flags) if flags else "python3"
    print(f"{label:<12} ->", (result.stdout or result.stderr).strip().splitlines()[-1])

python3      -> AssertionError: amount must be positive
python3 -O   -> balance after withdrawing -50: 150


Without the flag, the guard fires. With it, a withdrawal of minus fifty pounds **increases the balance to 150** and nothing complains. `-O` is used in production deployments precisely because people expect it to be harmless.

So the rule is:

| Use | For | Because |
|---|---|---|
| `assert` | things you believe are already true | it documents an assumption, and may vanish |
| `raise ValueError` | checking input from a user, a file, or a network | it always runs |

The same function, written properly:

In [15]:
def withdraw(balance: float, amount: float) -> float:
    if amount <= 0:
        raise ValueError(f"amount must be positive, got {amount}")
    return balance - amount


try:
    withdraw(100, -50)
except ValueError as e:
    print("ValueError:", e)

ValueError: amount must be positive, got -50


Where `assert` does belong is inside tests, which is the next section, and as an internal sanity check on something you have already validated.

---
# 6. Testing With `pytest`

A test is a small function that runs your code and checks the answer. `pytest` finds them and runs them for you.

Its conventions are the whole setup: files named `test_*.py`, functions named `test_*`, and plain `assert` for the checks. There is no class to inherit from and nothing to register.

Here is a test file for the `prices.py` we wrote in section 2:

In [16]:
%%writefile reliable_demo/test_prices.py
"""Tests for prices.py. Run with: pytest"""
import pytest

from prices import to_price


def test_plain_number():
    assert to_price("12.50") == 12.50


def test_strips_currency_and_separators():
    assert to_price("£1,299.00") == 1299.0


def test_missing_price_is_none():
    assert to_price("") is None
    assert to_price("free") is None


def test_none_input_is_none():
    assert to_price(None) is None

Writing reliable_demo/test_prices.py


Now run it. `pytest` prints how long it took, which changes on every run, so this notebook hides that one number to keep its output stable.

In [17]:
import re


def run_pytest(*args):
    result = subprocess.run([sys.executable, "-m", "pytest",
                             "--no-header", "--color=no",
                             "-p", "no:cacheprovider", *args],
                            cwd="reliable_demo", capture_output=True, text=True,
                            env={**os.environ, "COLUMNS": "80"})
    # the duration is different every run, so blank it out
    print(re.sub(r"in \d+\.\d+s", "in Xs", result.stdout))


run_pytest("test_prices.py", "-q")

....                                                                     [100%]
4 passed in Xs



Four dots, four passing tests. On the command line that is just:

```bash
pytest
```

### Reading a failure

A passing test tells you nothing you did not hope for. The useful output is a failing one, so here is a test with a wrong expectation in it:

In [18]:
%%writefile reliable_demo/test_broken.py
from prices import to_price, total


def test_total_of_two_prices():
    assert total([12.50, 8.00]) == 20.50


def test_price_keeps_the_pence():
    assert to_price("£19.99") == 19.0        # wrong on purpose

Writing reliable_demo/test_broken.py


In [19]:
run_pytest("test_broken.py", "-q")

.F                                                                       [100%]
=================================== FAILURES ===================================
__________________________ test_price_keeps_the_pence __________________________

    def test_price_keeps_the_pence():
>       assert to_price("£19.99") == 19.0        # wrong on purpose
E       AssertionError: assert 19.99 == 19.0
E        +  where 19.99 = to_price('£19.99')

test_broken.py:9: AssertionError
=========================== short test summary info ============================
FAILED test_broken.py::test_price_keeps_the_pence - AssertionError: assert 19...
1 failed, 1 passed in Xs



`pytest` shows the line that failed, then the values on both sides: `assert 19.99 == 19.0`. That second line is the reason people use `pytest` rather than writing asserts by hand, and it comes from nothing more than a plain `assert`.

### Testing that something raises

The other half of testing is checking that bad input is rejected. `pytest.raises` is a context manager, from chapter 14:

In [20]:
%%writefile reliable_demo/test_raises.py
import pytest


def withdraw(balance, amount):
    if amount <= 0:
        raise ValueError(f"amount must be positive, got {amount}")
    return balance - amount


def test_rejects_a_negative_amount():
    with pytest.raises(ValueError):
        withdraw(100, -50)


def test_says_what_was_wrong():
    with pytest.raises(ValueError, match="must be positive"):
        withdraw(100, 0)


def test_a_normal_withdrawal_works():
    assert withdraw(100, 30) == 70

Writing reliable_demo/test_raises.py


In [21]:
run_pytest("test_raises.py", "-q")

...                                                                      [100%]
3 passed in Xs



The test passes when the exception is raised, and fails when it is not, which is the opposite of everywhere else in this course.

### One test, many cases

Writing the same test five times with different numbers is a common way to make testing feel tedious. `@pytest.mark.parametrize` is a decorator, from chapter 13, that runs one test once per case:

In [22]:
%%writefile reliable_demo/test_many.py
import pytest

from prices import to_price


@pytest.mark.parametrize("text, expected", [
    ("12.50", 12.50),
    ("£12.50", 12.50),
    ("  12.50  ", 12.50),
    ("£1,299.00", 1299.0),
    ("USD 45", 45.0),
    ("", None),
    ("free", None),
    (None, None),
])
def test_to_price(text, expected):
    assert to_price(text) == expected

Writing reliable_demo/test_many.py


In [23]:
run_pytest("test_many.py", "-v")

============================= test session starts ==============================
collecting ... collected 8 items

test_many.py::test_to_price[12.50-12.5] PASSED                           [ 12%]
test_many.py::test_to_price[\xa312.50-12.5] PASSED                       [ 25%]
test_many.py::test_to_price[  12.50  -12.5] PASSED                       [ 37%]
test_many.py::test_to_price[\xa31,299.00-1299.0] PASSED                  [ 50%]
test_many.py::test_to_price[USD 45-45.0] PASSED                          [ 62%]
test_many.py::test_to_price[-None] PASSED                                [ 75%]
test_many.py::test_to_price[free-None] PASSED                            [ 87%]
test_many.py::test_to_price[None-None] PASSED                            [100%]

============================== 8 passed in Xs ===============================



Eight cases, one test function, and each case is named in the output so a failure tells you which input broke.

### Running everything

With no filename, `pytest` collects every `test_*.py` it can find. That includes the broken one, on purpose:

In [24]:
run_pytest("-q")

.F...............                                                        [100%]
=================================== FAILURES ===================================
__________________________ test_price_keeps_the_pence __________________________

    def test_price_keeps_the_pence():
>       assert to_price("£19.99") == 19.0        # wrong on purpose
E       AssertionError: assert 19.99 == 19.0
E        +  where 19.99 = to_price('£19.99')

test_broken.py:9: AssertionError
=========================== short test summary info ============================
FAILED test_broken.py::test_price_keeps_the_pence - AssertionError: assert 19...
1 failed, 16 passed in Xs



Three files pass, one fails, and the summary at the bottom names exactly what to look at.

### What is worth testing

You cannot test everything, and trying is how people give up on testing. Aim at:

- **The edge cases you already thought about.** Empty input, `None`, zero, a missing field. Chapter 18's missing price is a test.
- **Anything you have fixed once.** A test written the day you fix a bug is the one that stops it coming back.
- **The parts with rules in them.** Parsing, validation, calculation. Not the glue.

What not to test: the standard library, the network (fake it or use a saved response, as chapter 17 did with `sample_data/`), and things whose answer changes, like the current time.

---
# 7. When It Breaks Anyway

### Reading a traceback

A traceback reads **bottom to top**. The bottom is the error; above it is the call that caused it, and above that the call that caused that.

In [25]:
%%writefile reliable_demo/report.py
from prices import to_price


def load_prices(rows):
    return [to_price(row["price"]) for row in rows]


def report(rows):
    prices = load_prices(rows)
    return sum(prices) / len(prices)


print(report([{"price": "£12.50"}, {"price": "free"}]))

Writing reliable_demo/report.py


In [26]:
result = subprocess.run([sys.executable, "report.py"],
                        cwd="reliable_demo", capture_output=True, text=True)

# Python reports absolute paths; shorten them so this reads like your own screen
print(result.stderr.replace(str(Path.cwd()) + "/", ""))

Traceback (most recent call last):
  File "reliable_demo/report.py", line 13, in <module>
    print(report([{"price": "£12.50"}, {"price": "free"}]))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "reliable_demo/report.py", line 10, in report
    return sum(prices) / len(prices)
           ^^^^^^^^^^^
TypeError: unsupported operand type(s) for +: 'float' and 'NoneType'



Read it the way it happened:

1. The **last line** says what went wrong: `unsupported operand type(s) for +: 'int' and 'NoneType'`.
2. The **frame above it** shows `sum(prices)`, so `prices` contains a `None`.
3. The **frame above that** shows `report(...)` calling `load_prices`, which is where the `None` came from: `to_price("free")` returned it, exactly as its docstring said it would.

The fix is in `report`, not in `to_price`. Nearly every traceback you read will be like this: the error surfaces one or two frames away from the mistake.

Two habits make them much easier. Read the last line first, then look for the topmost frame **in your own code**, ignoring frames inside libraries. And do not paste the error into a search engine before reading it; it usually says precisely what is wrong.

### `breakpoint()`

Putting `print` everywhere works, and stopping the program to look around works better. `breakpoint()` drops you into the debugger at that line:

```python
def report(rows):
    prices = load_prices(rows)
    breakpoint()                  # execution stops here
    return sum(prices) / len(prices)
```

It needs a terminal, so it cannot run in this notebook, but in a script you get a `(Pdb)` prompt with the program paused and every local variable available.

| Command | Does |
|---|---|
| `p prices` | print an expression |
| `n` | run the next line |
| `s` | step into the function being called |
| `c` | continue until the next breakpoint or the end |
| `l` | list the code around where you are |
| `w` | show the stack, the same frames as a traceback |
| `q` | quit |

Typing `p prices` at that prompt would have shown `[12.5, None]` immediately, which is the whole investigation.

In VS Code or PyCharm the same thing is a red dot in the margin, and the variables appear in a panel. Either way, the skill is the same: stop close to the problem and look at the actual values.

---
# 8. PEP 8, Briefly

PEP 8 is Python's style guide. Following it matters for one reason: code that looks like every other Python project is code other people can read quickly.

The parts that come up constantly:

| | |
|---|---|
| Indentation | 4 spaces, never tabs |
| Line length | 79 characters, or whatever your team agreed |
| Functions and variables | `lower_case_with_underscores` |
| Classes | `CapWords` |
| Constants | `ALL_CAPS` |
| Imports | standard library, then third party, then yours, each group separated |
| Blank lines | two between top-level definitions, one between methods |
| Spaces | `x = 1` and `f(a, b)`, not `x=1` or `f( a,b )` |

Nobody applies this by hand. `black` reformats a file to a consistent style, and `ruff` finds the problems, both in under a second:

```bash
pip install black ruff
black your_package/
ruff check your_package/
```

Run them before you commit and the question never comes up again. The naming conventions are the part worth memorising, because no formatter will rename anything for you.

---
# 9. Common Mistakes

| Mistake | What happens | Fix |
|---|---|---|
| Expecting hints to be enforced | A wrong type runs happily | Run `mypy` |
| `assert` for validating input | `-O` deletes the check in production | `raise ValueError` |
| `print` for diagnostics | Nothing recorded, no levels, no source | `logging` |
| `except: pass` | The failure leaves no trace at all | `logger.exception(...)` |
| Logging a key or a token | Secrets end up in files and log services | Log the request, never the credential |
| `logger.error("failed")` | True, and useless | Log the values you will want |
| Testing only the happy path | The empty and `None` cases break in production | Test the edges first |
| Tests that call the network | Slow, and they fail when the site is down | Use a saved response |
| Reading a traceback top down | You start in library code you did not write | Last line first, then your own frames |

---
# 10. Summary: Your Reliability Cheat Sheet

**Type hints**

```python
def to_price(text: str) -> float | None: ...
def parse(rows: list[dict[str, str]]) -> dict[str, float]: ...
def save(path: str, rows: list[dict]) -> None: ...
```

```bash
pip install mypy
mypy your_package/            # hints are checked here, never at runtime
```

**Docstring**

```python
def to_price(text: str) -> float | None:
    """Return the number in a price string, or None if there is not one.

    Strips currency symbols and separators, so "£1,299.00" becomes 1299.0.

    Raises:
        TypeError: if text is neither a string nor None.
    """
```

**Logging**

```python
import logging
logger = logging.getLogger(__name__)

logger.debug("detail while developing")
logger.info("normal progress")
logger.warning("odd, but carried on")
logger.error("this piece of work failed")
logger.exception("same as error, plus the traceback")   # inside except only

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)-8s %(name)s | %(message)s")
```

**`assert` against `raise`**

```python
assert len(rows) > 0            # an assumption. -O deletes this.
if not rows:                    # a check. always runs.
    raise ValueError("no rows")
```

**pytest**

```python
# test_prices.py
import pytest
from prices import to_price

def test_plain_number():
    assert to_price("12.50") == 12.50

def test_rejects_negatives():
    with pytest.raises(ValueError, match="must be positive"):
        withdraw(100, -50)

@pytest.mark.parametrize("text, expected", [("12.50", 12.5), ("free", None)])
def test_cases(text, expected):
    assert to_price(text) == expected
```

```bash
pytest                  # everything
pytest -v               # one line per test
pytest test_prices.py   # one file
pytest -k price         # tests whose name contains "price"
pytest -x               # stop at the first failure
```

**Debugging**

```python
breakpoint()            # then: p name, n, s, c, l, w, q
```

Read tracebacks bottom to top. Last line first, then your own frames.

**Style**

```bash
black your_package/     # reformat
ruff check your_package/   # find problems
```

---

**Next:** chapter 20 is the capstone. One project, built with everything from chapters 1 to 19: fetched over HTTP with retries and pagination, cleaned with regex, validated with custom exceptions, modelled with classes, logged, type-hinted, tested, and saved as a tidy dataset ready for the NumPy and pandas repository.